# Document Loading

Note to students.
During periods of high load you may find the notebook unresponsive. It may appear to execute a cell, update the completion number in brackets [#] at the left of the cell but you may find the cell has not executed. This is particularly obvious on print statements when there is no output. If this happens, restart the kernel using the command under the Kernel tab.
## Retrieval augmented generation

In retrieval augmented generation (RAG), an LLM retrieves contextual documents from an external dataset as part of its execution.

This is useful if we want to ask question about specific documents (e.g., our PDFs, a set of videos, etc).

In [ ]:
# 基础环境准备：加载 API Key 等配置
import os
import openai
import sys
sys.path.append('../..')  # 把上级目录加入模块搜索路径，方便 import 课程自带的辅助模块

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())  # 读取同目录/上级目录下的 .env 文件，把里面的环境变量注入 os.environ

# 注意：这是 2023 年课程的旧写法。openai>=1.0 后官方推荐用 `client = OpenAI()` 这种 client 对象方式调用，
# 而不是直接给 openai 模块的 api_key 属性赋值。不过这里只是给模块设个全局默认 key，
# 后面 LangChain 的 OpenAIEmbeddings / ChatOpenAI 实际上会自己从环境变量 OPENAI_API_KEY 里读取 key，
# 所以这行本身不会报错，可以保留，但对本notebook的其余代码其实不是必需的。
openai.api_key  = os.environ['OPENAI_API_KEY']

PDFs
Let's load a PDF transcript from Andrew Ng's famous CS229 course! These documents are the result of automated transcription so words and sentences are sometimes split unexpectedly.

In [ ]:
# 课程平台已经预装了这些包，这里的 pip install 只是给你在自己电脑上跑时的提示，不需要真的执行。
# pypdf 是 PyPDFLoader 底层用来解析 PDF 文件的库。
#! pip install pypdf

In [ ]:
# PyPDFLoader：RAG 流程第一步——文档加载（Document Loading）
# 它会把 PDF 逐页解析成一个 Document 对象列表，每个 Document 有 page_content（该页文本）和 metadata（来源、页码等）
#
# 注意：这里用的是新版路径 langchain_community.document_loaders。
# 课程原视频年代（2023）用的是旧版写法 `from langchain.document_loaders import PyPDFLoader`，
# 在当前安装的 langchain 1.4.0 下，document_loaders 已经从主包 langchain 拆分到独立的 langchain_community 包，
# 旧路径会直接 ModuleNotFoundError，必须改成下面这种新路径。
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf")
pages = loader.load()  # 触发真正的 PDF 解析，返回 List[Document]，每个元素对应 PDF 的一页

In [ ]:
# pages 是一个列表，长度等于 PDF 的总页数
len(pages)

In [ ]:
# 取第一页看看 Document 对象长什么样
page=pages[0]

In [ ]:
# page_content 是这一页解析出来的纯文本内容，这里只打印前 500 个字符看个大概
print(page.page_content[0:500])

In [ ]:
# metadata 里通常包含 source（文件路径）和 page（页码，从 0 开始计数）
# 这些元数据在后面做向量检索时非常有用，可以用来做 metadata filter 或者展示引用来源
page.metadata

In [ ]:
# YouTube 音频转文本加载：GenericLoader 是一个通用框架，
# 由 BlobLoader（负责“找到原始数据在哪”，比如本地文件系统或 YouTube）
# 和 BlobParser（负责“把原始数据解析成文本”，这里用 OpenAI 的 Whisper 语音转文字模型）组合而成。
# 同样是新版 langchain_community 路径，旧版是 langchain.document_loaders.generic 等。
from langchain_community.document_loaders.generic import GenericLoader,FileSystemBlobLoader
from langchain_community.document_loaders.parsers import OpenAIWhisperParser
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader

**RAG 流程小结**：本 notebook 展示的是 RAG（检索增强生成）流水线的第一步——**文档加载（Document Loading）**。不同来源的数据（PDF、YouTube 音频、网页、Notion 笔记）都会被统一加载成 LangChain 的 `Document` 对象（`page_content` + `metadata`），后续章节会对这些 `Document` 做切分（Splitting）、向量化（Embedding）、存入向量库（Vectorstore），最终支持检索问答。

In [ ]:
# 这里演示两种获取音频的方式：
# 1) YoutubeAudioLoader([url], save_dir) —— 直接从 YouTube 拉取音频（需要真实网络，课程里默认注释掉）
# 2) FileSystemBlobLoader(save_dir, glob="*.m4a") —— 读本地已经下载好的音频文件（*.m4a 是苹果的音频格式）
# OpenAIWhisperParser 负责把音频转成文字（语音识别），最终 docs 里每个元素是转录出来的一段文本
url="https://www.youtube.com/watch?v=jGwO_UgTS7I"
save_dir="docs/youtube/"
loader = GenericLoader(
    #YoutubeAudioLoader([url],save_dir),  # fetch from youtube
    FileSystemBlobLoader(save_dir, glob="*.m4a"),   #fetch locally
    OpenAIWhisperParser()
)
docs = loader.load()

In [ ]:
# 看看语音转录出来的文本前 500 个字符
docs[0].page_content[0:500]

In [ ]:
# WebBaseLoader：直接抓取网页正文作为文档来源（这里是一篇 GitHub 上的 Markdown 文档）
# 同样使用新版 langchain_community 路径
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://github.com/basecamp/handbook/blob/master/titles-for-programmers.md")

In [ ]:
# load() 会真正发起网络请求抓取网页并解析 HTML 为纯文本
docs = loader.load()

In [ ]:
print(docs[0].page_content[:500])

In [ ]:
# NotionDirectoryLoader：加载从 Notion 导出的 Markdown 文件目录（每个页面导出成一个 .md 文件）
# 常用于把团队的 Notion 知识库接入 RAG 系统
from langchain_community.document_loaders import NotionDirectoryLoader
loader = NotionDirectoryLoader("docs/Notion_DB")
docs = loader.load()

In [ ]:
print(docs[0].page_content[0:200])

In [ ]:
# Notion 加载出来的 metadata 通常只有 source（文件路径），不像 PDF 那样有页码
docs[0].metadata